# 02 — CNN Training with Ray Data + Ray Train + Lance

**Purpose:** Train a CNN image classifier on the synthetic Lance dataset built in `01_create_lance_dataset.ipynb`, using `ray.data.read_lance` for data loading and Ray Train for distributed training. This is the happy path — the idiomatic Lance + Ray training workflow on Databricks, end to end.

---

## Why Ray Data + Ray Train?

Ray Data's in-memory engine represents data as Apache Arrow blocks, and Lance stores data in Arrow IPC format natively — so Ray Data reads directly into Arrow blocks with no deserialization step. That CPU budget goes to image decode and augmentation instead.

Three properties make this stack fit distributed image training:

**1. Arrow-native reads.** `ray.data.read_lance` loads straight into Arrow blocks. No conversion tax on every batch.

**2. O(1) random access.** A Lance dataset is composed of independent fragments, each row addressable by a direct byte-offset seek. Shuffled batch reads fetch exactly the rows requested, at constant cost regardless of dataset size.

**3. Fragment-parallel reads align to Ray's actor model.** Each Ray worker actor reads its assigned fragment(s) with no cross-worker coordination. Object storage pressure is predictable and coalesced — no thundering herd of per-image GET requests.

---

## What this notebook does

1. **Load the Lance dataset.** Open the Lance `frames` dataset from `01_create_lance_dataset.ipynb`. Pin the dataset version for training reproducibility — any future appends or schema updates won't affect this run.

2. **Build the Ray Data pipeline.** Use `ray.data.read_lance` with column projection (`image`, `category` only — caption, embedding, and metadata columns are not loaded during training). Define preprocessing transforms as Ray Data map operations: JPEG decode → resize to model input resolution → normalize → random horizontal flip. Preprocessing runs on CPU actors in parallel with GPU training, keeping the GPU saturated.

3. **Define the model.** A ResNet-50 image classifier over the synthetic dataset's `category` labels. Loss: cross-entropy.

4. **Distributed training with Ray Train.** Configure a `TorchTrainer` with DDP (DistributedDataParallel) across all GPU workers in the Ray cluster. Each worker receives its shard of the Lance dataset via Ray Data — no manual data partitioning. Checkpoint model weights to UC Volumes at configurable intervals.

5. **Log to MLflow.** Track loss, accuracy, and data-loading throughput (samples/sec) per epoch. Pin the Lance dataset version as a run parameter so training runs are fully reproducible.

---

**Inputs:** Lance `frames` dataset at `/Volumes/{catalog}/{schema}/{volume}/frames/` (inline JPEG bytes + `category` labels)

**Outputs:** Trained ResNet-50 classifier logged to MLflow